# Explore Negative Underlying Load

This notebook investigates selected Solar Analytics CICCADA BESS households where the computed underlying load is meaningfully negative. The focus is diagnostic rather than corrective: the plots are intended to show whether the negative values line up with the corrected net-load signal, PV generation, battery storage behavior, or polarity assumptions.

The notebook uses the already processed selected 100-household sample and does not regenerate any datasets.

## 1. Purpose And Signal Definitions

This section records the signal convention used by the current processing workflow. The next code cell defines the core constants so every later plot uses the same threshold, profile name, and component formulas.

Signal convention used here:

- Selected profile: `current_polarity_adjusted`.
- Meaningful negative underlying-load threshold: `underlying_load_kW < -0.05`.
- `net_load_with_pv_and_battery_kW` is the corrected `ac_load_net` signal.
- `pv_generation_kW = underlying_load_kW - net_load_with_pv_kW`.
- `battery_storage_kW = net_load_with_pv_and_battery_kW - net_load_with_pv_kW`.
- Positive `battery_storage_kW` means the battery is charging.

In [ ]:
# Purpose: keep the diagnostic profile, threshold, and formulas explicit.
# These constants should match the current SA BESS publication workflow outputs.

EXPECTED_SIGNAL_PROFILE = "current_polarity_adjusted"
EXPECTED_SELECTED_HOUSEHOLDS = 100
EXPECTED_NEGATIVE_HOUSEHOLDS = 56
NEGATIVE_THRESHOLD_KW = -0.05
FIXED_AEST_OFFSET_HOURS = 10

TARGET_COLUMNS = [
    "underlying_load_kW",
    "net_load_with_pv_kW",
    "net_load_with_pv_and_battery_kW",
]

COMPONENT_COLUMNS = [
    "underlying_load_kW",
    "net_load_with_pv_and_battery_kW",
    "pv_generation_kW",
    "battery_storage_kW",
]

COMPONENT_LABELS = {
    "underlying_load_kW": "Underlying load",
    "net_load_with_pv_kW": "Net load with PV",
    "net_load_with_pv_and_battery_kW": "Corrected net load with PV and battery",
    "pv_generation_kW": "PV generation",
    "battery_storage_kW": "Battery storage, positive = charging",
    "battery_net_discharge_kW": "Battery net discharge",
}

COMPONENT_COLORS = {
    "underlying_load_kW": "#2f5597",
    "net_load_with_pv_and_battery_kW": "#4b5563",
    "pv_generation_kW": "#f2b705",
    "battery_storage_kW": "#8c564b",
    "battery_net_discharge_kW": "#9467bd",
}

## 2. Setup And Dependency Check

This cell imports the analysis libraries, finds the repository root, checks the expected cleaned-data paths, and tests whether Plotly and ipywidgets are installed. If Plotly is missing, the notebook still loads and validates the data, but the interactive figure cells will print installation guidance instead of failing noisily.

In [ ]:
# Purpose: import dependencies, discover paths, and report optional plotting support.
# Plotly and ipywidgets are intentionally optional because they are not tracked dependencies here.

import os
from pathlib import Path
import importlib.util
import warnings

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
try:
    from IPython.display import HTML, Markdown, display
except ModuleNotFoundError:
    HTML = str

    class Markdown(str):
        pass

    def display(obj):
        print(obj)

warnings.filterwarnings("ignore", category=FutureWarning)

PLOTLY_AVAILABLE = importlib.util.find_spec("plotly") is not None
WIDGETS_AVAILABLE = importlib.util.find_spec("ipywidgets") is not None

if PLOTLY_AVAILABLE:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
else:
    px = go = make_subplots = None

if WIDGETS_AVAILABLE:
    import ipywidgets as widgets
else:
    widgets = None


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until the PyNNLF repository root is found."""
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists() or (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root from the current working directory.")


REPO_ROOT = find_repo_root()
def raw_data_root():
    """Locate the local source data tree, which is not distributed with this repository.

    Set the PYNNLF_RAW_DATA_DIR environment variable to the directory holding the
    "1. raw", "2. processed" and "3. cleaned" folders before running this notebook.

    Returns:
        Path: root of the local source data tree.
    """
    root = os.environ.get("PYNNLF_RAW_DATA_DIR")
    if not root:
        raise RuntimeError(
            "PYNNLF_RAW_DATA_DIR is not set. Point it at your local source data "
            "directory; see the Data section of the repository README."
        )
    return Path(root)


RAW_DATA_ROOT = raw_data_root()

CLEANED_SA_BESS_DIR = RAW_DATA_ROOT / "3. cleaned" / "SA BESS"
INTERMEDIATE_DIR = CLEANED_SA_BESS_DIR / "intermediate"
PROCESSED_DIR = CLEANED_SA_BESS_DIR / "processed"
DIAGNOSTICS_DIR = CLEANED_SA_BESS_DIR / "diagnostics"

SUMMARY_PATH = PROCESSED_DIR / "sa_bess_selected_household_summary.csv"
SITE_TIMESERIES_PATH = INTERMEDIATE_DIR / "sa_bess_selected_site_timeseries_5min.parquet"
SOURCE_HISTORY_PATH = INTERMEDIATE_DIR / "sa_bess_selected_source_history.parquet"
DIAGNOSTIC_SCORE_PATH = DIAGNOSTICS_DIR / "sa_bess_signal_diagnostic_scores.csv"

RAW_BESS_DIR = RAW_DATA_ROOT / "1. raw" / "Solar Analytics Data from CICCADA" / "bess_data"
CIRCUIT_META_PATH = RAW_BESS_DIR / "circuit_meta_data.csv"

REQUIRED_PATHS = [SUMMARY_PATH, SITE_TIMESERIES_PATH, SOURCE_HISTORY_PATH, DIAGNOSTIC_SCORE_PATH]
missing_paths = [path for path in REQUIRED_PATHS if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Missing expected cleaned SA BESS outputs:\n" + "\n".join(map(str, missing_paths)))

print(f"Repository root: {REPO_ROOT}")
print(f"Cleaned SA BESS folder: {CLEANED_SA_BESS_DIR}")
print(f"Plotly available: {PLOTLY_AVAILABLE}")
print(f"ipywidgets available: {WIDGETS_AVAILABLE}")

if not PLOTLY_AVAILABLE or not WIDGETS_AVAILABLE:
    display(Markdown(
        "**Interactive dependency note:** Plotly and ipywidgets are optional for this repository. "
        "To enable every interactive cell in this notebook, install them in the active notebook environment, "
        "for example `python -m pip install plotly ipywidgets`. No tracked dependency files are changed by this notebook."
    ))

## 3. Load Processed Selected-Household Data

This section loads the selected-household summary and then reads only the 56 negative households from the 5-minute selected-site parquet. Reading only the negative households keeps the notebook responsive while preserving the full 5-minute detail needed for diagnosis.

In [ ]:
# Purpose: load selected-household metadata and the negative-household 5-minute timeseries.
# The parquet read is filtered by site_id so this notebook does not need all 100 households in memory.


def parse_mixed_datetime(series: pd.Series) -> pd.Series:
    """Parse the date strings written by the SA BESS workflow, preserving day-first dates."""
    return pd.to_datetime(series, errors="coerce", dayfirst=True)


summary = pd.read_csv(SUMMARY_PATH)
for column in ["selected_overlap_start", "selected_overlap_end", "first_observed_date", "monitoring_start"]:
    if column in summary.columns:
        summary[column] = parse_mixed_datetime(summary[column])

signal_profiles = sorted(summary["formula_profile"].dropna().unique().tolist())
if signal_profiles != [EXPECTED_SIGNAL_PROFILE]:
    raise ValueError(f"Expected only {EXPECTED_SIGNAL_PROFILE}, found {signal_profiles}")

summary["site_id"] = summary["site_id"].astype("int64")
summary["selection_rank"] = summary["selection_rank"].astype("int64")
summary["negative_count"] = summary["meaningful_negative_underlying_load_count_post_fill"].astype("int64")
summary["negative_pct"] = summary["meaningful_negative_underlying_load_pct_post_fill"].astype(float)
summary["min_underlying_load_kW"] = summary["min_underlying_load_kW_post_fill"].astype(float)

negative_summary = (
    summary.loc[summary["negative_count"].gt(0)]
    .sort_values(["negative_count", "min_underlying_load_kW"], ascending=[False, True])
    .reset_index(drop=True)
)
negative_site_ids = [int(site_id) for site_id in negative_summary["site_id"]]

if len(summary) != EXPECTED_SELECTED_HOUSEHOLDS:
    raise AssertionError(f"Expected {EXPECTED_SELECTED_HOUSEHOLDS} selected households, found {len(summary)}")
if len(negative_summary) != EXPECTED_NEGATIVE_HOUSEHOLDS:
    raise AssertionError(f"Expected {EXPECTED_NEGATIVE_HOUSEHOLDS} negative households, found {len(negative_summary)}")

site_dataset = ds.dataset(str(SITE_TIMESERIES_PATH), format="parquet")
source_dataset = ds.dataset(str(SOURCE_HISTORY_PATH), format="parquet")

site_table = site_dataset.to_table(
    columns=["site_id", "datetime", *TARGET_COLUMNS],
    filter=ds.field("site_id").isin(negative_site_ids),
)
negative_ts = site_table.to_pandas().sort_values(["site_id", "datetime"]).reset_index(drop=True)
negative_ts["site_id"] = negative_ts["site_id"].astype("int64")
negative_ts["datetime"] = pd.to_datetime(negative_ts["datetime"])

print(f"Selected households: {len(summary):,}")
print(f"Negative households: {len(negative_summary):,}")
print(f"Negative-household 5-minute rows loaded: {len(negative_ts):,}")
print(f"Date range: {negative_ts['datetime'].min()} to {negative_ts['datetime'].max()}")
display(negative_summary[[
    "selection_rank",
    "site_id",
    "state",
    "postcode",
    "negative_count",
    "negative_pct",
    "min_underlying_load_kW",
    "dc_capacity_kw",
    "ac_capacity_kw",
]].head(12))

## 4. Reconstruct Components And Validate Identities

The processed selected-site parquet contains the three target series. This section reconstructs PV generation and battery storage from those series, then validates the arithmetic identities that connect underlying load, net load, PV, and battery behavior.

In [ ]:
# Purpose: reconstruct components and verify that the formulas match the processed target series.
# Positive battery_storage_kW means charging because it raises net load with battery above net load with PV.

negative_ts = negative_ts.copy()
negative_ts["pv_generation_kW"] = negative_ts["underlying_load_kW"] - negative_ts["net_load_with_pv_kW"]
negative_ts["battery_storage_kW"] = negative_ts["net_load_with_pv_and_battery_kW"] - negative_ts["net_load_with_pv_kW"]
negative_ts["battery_net_discharge_kW"] = -negative_ts["battery_storage_kW"]
negative_ts["is_negative_underlying"] = negative_ts["underlying_load_kW"].lt(NEGATIVE_THRESHOLD_KW)
negative_ts["negative_label"] = np.where(
    negative_ts["is_negative_underlying"],
    f"underlying < {NEGATIVE_THRESHOLD_KW} kW",
    f"underlying >= {NEGATIVE_THRESHOLD_KW} kW",
)

# Time features are reused by the weekly-profile and heatmap diagnostics.
negative_ts["weekday"] = negative_ts["datetime"].dt.dayofweek
negative_ts["weekday_name"] = negative_ts["datetime"].dt.day_name().str[:3]
negative_ts["hour"] = negative_ts["datetime"].dt.hour
negative_ts["minute"] = negative_ts["datetime"].dt.minute
negative_ts["week_slot_5min"] = negative_ts["weekday"] * 288 + negative_ts["hour"] * 12 + negative_ts["minute"] // 5
negative_ts["hour_of_week"] = negative_ts["weekday"] * 24 + negative_ts["hour"]
negative_ts["time_of_day"] = negative_ts["datetime"].dt.strftime("%H:%M")

pv_identity_error = (
    negative_ts["underlying_load_kW"] - negative_ts["pv_generation_kW"] - negative_ts["net_load_with_pv_kW"]
).abs().max()
battery_identity_error = (
    negative_ts["net_load_with_pv_and_battery_kW"] - negative_ts["battery_storage_kW"] - negative_ts["net_load_with_pv_kW"]
).abs().max()

summary_negative_total = int(negative_summary["negative_count"].sum())
loaded_negative_total = int(negative_ts["is_negative_underlying"].sum())
if summary_negative_total != loaded_negative_total:
    raise AssertionError(
        f"Summary negative count ({summary_negative_total:,}) does not match loaded rows ({loaded_negative_total:,})."
    )

if pv_identity_error > 1e-9 or battery_identity_error > 1e-9:
    raise AssertionError(
        "Component identity check failed: "
        f"pv_error={pv_identity_error:.3e}, battery_error={battery_identity_error:.3e}"
    )

print(f"PV identity max absolute error: {pv_identity_error:.3e} kW")
print(f"Battery identity max absolute error: {battery_identity_error:.3e} kW")
print(f"Total meaningful negative 5-minute rows: {loaded_negative_total:,}")
print(f"Share of loaded negative-household rows below threshold: {100 * loaded_negative_total / len(negative_ts):.2f}%")

## 5. Negative-Household Overview

This section gives a high-level view of which households drive the negative-underlying behavior. The table and bar chart rank households by frequency of negative intervals, while the component summary gives a first screening hint about whether the negative values tend to coincide with corrected net-load export, PV behavior, or battery charging/discharging.

In [ ]:
# Purpose: build a ranked household table plus simple component screening hints.
# The heuristic label is a triage aid only; use the later plots before changing any signal assumptions.

negative_only = negative_ts.loc[negative_ts["is_negative_underlying"]].copy()

component_screen = (
    negative_only.groupby("site_id", observed=True)
    .agg(
        negative_rows=("underlying_load_kW", "size"),
        min_underlying_kW=("underlying_load_kW", "min"),
        median_underlying_kW=("underlying_load_kW", "median"),
        median_corrected_net_kW=("net_load_with_pv_and_battery_kW", "median"),
        median_pv_generation_kW=("pv_generation_kW", "median"),
        median_battery_storage_kW=("battery_storage_kW", "median"),
        pct_corrected_net_negative=("net_load_with_pv_and_battery_kW", lambda s: 100 * s.lt(0).mean()),
        pct_pv_negative=("pv_generation_kW", lambda s: 100 * s.lt(-0.05).mean()),
        pct_battery_charging=("battery_storage_kW", lambda s: 100 * s.gt(0.05).mean()),
        pct_battery_discharging=("battery_storage_kW", lambda s: 100 * s.lt(-0.05).mean()),
    )
    .reset_index()
)


def screening_hint(row: pd.Series) -> str:
    """Return a conservative diagnostic hint for negative intervals at one site."""
    if row["pct_pv_negative"] >= 25:
        return "PV sign or night offset check"
    if row["pct_corrected_net_negative"] >= 50:
        return "Corrected net load/export dominant"
    if row["pct_battery_charging"] >= 50 and row["median_battery_storage_kW"] > 0:
        return "Battery charging subtraction dominant"
    if row["pct_battery_discharging"] >= 50:
        return "Battery discharge sign check"
    return "Mixed components"


component_screen["screening_hint"] = component_screen.apply(screening_hint, axis=1)
ranked_negative = (
    negative_summary.merge(component_screen, on="site_id", how="left")
    .sort_values(["negative_count", "min_underlying_load_kW"], ascending=[False, True])
    .reset_index(drop=True)
)
ranked_negative["rank_by_negative_count"] = np.arange(1, len(ranked_negative) + 1)

overview_columns = [
    "rank_by_negative_count",
    "selection_rank",
    "site_id",
    "state",
    "postcode",
    "negative_count",
    "negative_pct",
    "min_underlying_load_kW",
    "median_corrected_net_kW",
    "median_pv_generation_kW",
    "median_battery_storage_kW",
    "pct_corrected_net_negative",
    "pct_pv_negative",
    "pct_battery_charging",
    "pct_battery_discharging",
    "screening_hint",
]

display(ranked_negative[overview_columns].round(3))

if PLOTLY_AVAILABLE:
    fig = px.bar(
        ranked_negative,
        x="site_id",
        y="negative_pct",
        color="screening_hint",
        hover_data=[
            "selection_rank",
            "negative_count",
            "min_underlying_load_kW",
            "median_corrected_net_kW",
            "median_pv_generation_kW",
            "median_battery_storage_kW",
        ],
        title="Negative underlying-load frequency by selected household",
        labels={"site_id": "Site ID", "negative_pct": "Negative intervals (%)"},
    )
    fig.update_xaxes(type="category")
    fig.update_layout(height=520, legend_title_text="Screening hint")
    fig.show()
else:
    print("Install plotly to render the ranked negative-household bar chart.")

In [ ]:
# Purpose: show when negative intervals occur across households and the weekly clock.
# The first heatmap is site by hour-of-week; the second aggregates events by weekday and hour.

if PLOTLY_AVAILABLE:
    site_hour = (
        negative_ts.groupby(["site_id", "hour_of_week"], observed=True)["is_negative_underlying"]
        .mean()
        .mul(100)
        .reset_index(name="negative_pct")
    )
    site_label_map = {
        int(row.site_id): f"rank {int(row.rank_by_negative_count):02d} | site {int(row.site_id)}"
        for row in ranked_negative.itertuples(index=False)
    }
    site_hour["site_label"] = site_hour["site_id"].map(site_label_map)
    heatmap = site_hour.pivot(index="site_label", columns="hour_of_week", values="negative_pct").fillna(0)
    heatmap = heatmap.reindex([site_label_map[int(site_id)] for site_id in ranked_negative["site_id"]])

    week_tick_values = [day * 24 + 12 for day in range(7)]
    week_tick_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

    fig = go.Figure(
        data=go.Heatmap(
            z=heatmap.to_numpy(),
            x=heatmap.columns,
            y=heatmap.index,
            colorscale="Magma",
            colorbar={"title": "Negative intervals (%)"},
            hovertemplate="%{y}<br>hour of week=%{x}<br>negative=%{z:.2f}%<extra></extra>",
        )
    )
    fig.update_layout(
        title="Negative underlying-load share by household and hour of week",
        height=max(520, 18 * len(heatmap.index)),
        xaxis_title="Hour of week",
        yaxis_title="Household",
    )
    fig.update_xaxes(tickmode="array", tickvals=week_tick_values, ticktext=week_tick_labels)
    fig.show()

    weekday_hour = (
        negative_ts.groupby(["weekday", "hour"], observed=True)["is_negative_underlying"]
        .mean()
        .mul(100)
        .reset_index(name="negative_pct")
    )
    weekday_hour["weekday_name"] = weekday_hour["weekday"].map(dict(enumerate(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])))
    weekly_heatmap = weekday_hour.pivot(index="weekday_name", columns="hour", values="negative_pct").reindex(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])

    fig = px.imshow(
        weekly_heatmap,
        aspect="auto",
        color_continuous_scale="Magma",
        labels={"x": "Hour of day", "y": "Day of week", "color": "Negative intervals (%)"},
        title="Aggregate negative underlying-load share by day of week and hour",
    )
    fig.update_layout(height=420)
    fig.show()
else:
    print("Install plotly to render the negative-event heatmaps.")

## 6. Typical Weekly Profiles

These plots average the full selected period into a typical week. The first view aggregates all negative households together, while the second view shows per-household percentile bands so outlier sites can be separated from the typical negative-household shape.

In [ ]:
# Purpose: compute and plot aggregate component profiles across the negative-household cohort.
# Aggregate profiles sum households at each timestamp first, then average each 5-minute slot of the week.

DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
WEEK_SLOT_TICKS = [day * 288 + 144 for day in range(7)]
WEEK_SLOT_LABELS = DAYS

negative_aggregate = (
    negative_ts.groupby("datetime", as_index=False, observed=True)[COMPONENT_COLUMNS]
    .sum()
    .sort_values("datetime")
)
negative_aggregate["week_slot_5min"] = (
    negative_aggregate["datetime"].dt.dayofweek * 288
    + negative_aggregate["datetime"].dt.hour * 12
    + negative_aggregate["datetime"].dt.minute // 5
)
aggregate_week = (
    negative_aggregate.groupby("week_slot_5min", observed=True)[COMPONENT_COLUMNS]
    .mean()
    .reindex(range(7 * 288))
    .reset_index()
)

if PLOTLY_AVAILABLE:
    fig = go.Figure()
    for column in COMPONENT_COLUMNS:
        fig.add_trace(
            go.Scatter(
                x=aggregate_week["week_slot_5min"],
                y=aggregate_week[column],
                mode="lines",
                name=COMPONENT_LABELS[column],
                line={"width": 2.2, "color": COMPONENT_COLORS.get(column)},
            )
        )
    for day in range(8):
        fig.add_vline(x=day * 288, line_width=0.7, line_color="rgba(120,120,120,0.35)")
    fig.add_hline(y=0, line_width=0.8, line_color="rgba(60,60,60,0.7)")
    fig.update_layout(
        title="Typical weekly aggregate profile across negative households",
        height=520,
        xaxis_title="Typical week",
        yaxis_title="Aggregate power (kW)",
        legend_title_text="Component",
    )
    fig.update_xaxes(tickmode="array", tickvals=WEEK_SLOT_TICKS, ticktext=WEEK_SLOT_LABELS)
    fig.show()
else:
    display(aggregate_week.head())
    print("Install plotly to render the typical weekly aggregate profile.")

In [ ]:
# Purpose: compare household-level weekly profiles using median and percentile bands.
# This makes it easier to see whether negative load is broad across sites or driven by extreme households.

site_weekly = (
    negative_ts.groupby(["site_id", "week_slot_5min"], observed=True)[COMPONENT_COLUMNS]
    .mean()
    .reset_index()
)
underlying_band = (
    site_weekly.groupby("week_slot_5min", observed=True)["underlying_load_kW"]
    .quantile([0.10, 0.50, 0.90])
    .unstack()
    .rename(columns={0.10: "p10", 0.50: "median", 0.90: "p90"})
    .reindex(range(7 * 288))
    .reset_index()
)
component_medians = (
    site_weekly.groupby("week_slot_5min", observed=True)[COMPONENT_COLUMNS]
    .median()
    .reindex(range(7 * 288))
    .reset_index()
)

if PLOTLY_AVAILABLE:
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=underlying_band["week_slot_5min"],
            y=underlying_band["p90"],
            mode="lines",
            line={"width": 0},
            showlegend=False,
            hoverinfo="skip",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=underlying_band["week_slot_5min"],
            y=underlying_band["p10"],
            mode="lines",
            fill="tonexty",
            fillcolor="rgba(47,85,151,0.18)",
            line={"width": 0},
            name="Underlying p10-p90 across households",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=underlying_band["week_slot_5min"],
            y=underlying_band["median"],
            mode="lines",
            name="Underlying median across households",
            line={"color": COMPONENT_COLORS["underlying_load_kW"], "width": 2.4},
        )
    )
    for column in ["net_load_with_pv_and_battery_kW", "pv_generation_kW", "battery_storage_kW"]:
        fig.add_trace(
            go.Scatter(
                x=component_medians["week_slot_5min"],
                y=component_medians[column],
                mode="lines",
                name=f"{COMPONENT_LABELS[column]} median",
                line={"width": 1.8, "color": COMPONENT_COLORS.get(column)},
            )
        )
    for day in range(8):
        fig.add_vline(x=day * 288, line_width=0.7, line_color="rgba(120,120,120,0.35)")
    fig.add_hline(y=NEGATIVE_THRESHOLD_KW, line_dash="dash", line_width=1.0, line_color="firebrick")
    fig.update_layout(
        title="Typical weekly household profile bands across negative households",
        height=520,
        xaxis_title="Typical week",
        yaxis_title="Per-household power (kW)",
        legend_title_text="Series",
    )
    fig.update_xaxes(tickmode="array", tickvals=WEEK_SLOT_TICKS, ticktext=WEEK_SLOT_LABELS)
    fig.show()
else:
    display(underlying_band.head())
    print("Install plotly to render the household profile percentile bands.")

## 7. Distribution Diagnostics

This section compares component distributions during negative and non-negative intervals. The sample size is capped before plotting so the notebook remains responsive while still showing the shape of the full 5-minute data.

In [ ]:
# Purpose: create distribution plots for the reconstructed components.
# Sampling is only for plotting speed; all ranked tables and validation checks above used the full loaded data.

DISTRIBUTION_SAMPLE_ROWS = 300_000
PLOT_RANDOM_SEED = 42

if len(negative_ts) > DISTRIBUTION_SAMPLE_ROWS:
    distribution_sample = negative_ts.sample(DISTRIBUTION_SAMPLE_ROWS, random_state=PLOT_RANDOM_SEED).copy()
else:
    distribution_sample = negative_ts.copy()

distribution_long = distribution_sample.melt(
    id_vars=["site_id", "datetime", "is_negative_underlying", "negative_label"],
    value_vars=COMPONENT_COLUMNS,
    var_name="component",
    value_name="power_kW",
)
distribution_long["component_label"] = distribution_long["component"].map(COMPONENT_LABELS)

if PLOTLY_AVAILABLE:
    fig = px.histogram(
        distribution_long,
        x="power_kW",
        color="negative_label",
        facet_row="component_label",
        nbins=140,
        histnorm="probability density",
        barmode="overlay",
        opacity=0.58,
        title="Component distributions during negative vs non-negative intervals",
        labels={"power_kW": "Power (kW)", "negative_label": "Interval class"},
        height=900,
    )
    fig.update_yaxes(matches=None)
    fig.add_vline(x=0, line_width=0.8, line_color="rgba(60,60,60,0.7)")
    fig.show()

    fig = px.violin(
        distribution_long,
        x="component_label",
        y="power_kW",
        color="negative_label",
        box=True,
        points=False,
        title="Component spread by interval class",
        labels={"component_label": "Component", "power_kW": "Power (kW)", "negative_label": "Interval class"},
        height=580,
    )
    fig.update_xaxes(tickangle=20)
    fig.show()
else:
    display(distribution_long.groupby(["component_label", "negative_label"])["power_kW"].describe().round(3))
    print("Install plotly to render the distribution diagnostics.")

In [ ]:
# Purpose: test whether negative underlying load aligns with net-load export, PV magnitude, or battery storage.
# The heatmaps use sampled rows so dense relationships are visible without drawing millions of markers.

SCATTER_SAMPLE_ROWS = 120_000
scatter_sample = distribution_sample
if len(scatter_sample) > SCATTER_SAMPLE_ROWS:
    scatter_sample = scatter_sample.sample(SCATTER_SAMPLE_ROWS, random_state=PLOT_RANDOM_SEED).copy()

if PLOTLY_AVAILABLE:
    fig = make_subplots(
        rows=1,
        cols=3,
        subplot_titles=(
            "Corrected net load vs underlying",
            "PV generation vs underlying",
            "Battery storage vs underlying",
        ),
        shared_yaxes=True,
    )
    pairs = [
        ("net_load_with_pv_and_battery_kW", "Corrected net load with PV and battery (kW)"),
        ("pv_generation_kW", "PV generation (kW)"),
        ("battery_storage_kW", "Battery storage, positive = charging (kW)"),
    ]
    for col_index, (x_column, x_title) in enumerate(pairs, start=1):
        fig.add_trace(
            go.Histogram2d(
                x=scatter_sample[x_column],
                y=scatter_sample["underlying_load_kW"],
                colorscale="Viridis",
                showscale=(col_index == 3),
                colorbar={"title": "Rows"} if col_index == 3 else None,
                nbinsx=80,
                nbinsy=80,
            ),
            row=1,
            col=col_index,
        )
        fig.add_hline(y=NEGATIVE_THRESHOLD_KW, line_dash="dash", line_color="firebrick", row=1, col=col_index)
        fig.update_xaxes(title_text=x_title, row=1, col=col_index)
    fig.update_yaxes(title_text="Underlying load (kW)", row=1, col=1)
    fig.update_layout(title="Component relationships in sampled 5-minute rows", height=520)
    fig.show()
else:
    relation_stats = scatter_sample[[
        "underlying_load_kW",
        "net_load_with_pv_and_battery_kW",
        "pv_generation_kW",
        "battery_storage_kW",
    ]].corr()
    display(relation_stats.round(3))
    print("Install plotly to render the component relationship heatmaps.")

## 8. Per-Household Drilldown

This section provides a granular view for one household at a time. The helper functions below load source-history slices when needed and build a Plotly time-series view with negative-underlying intervals highlighted.

In [ ]:
# Purpose: define reusable drilldown helpers for a single selected household.
# Source-history parquet reads are filtered by site/date so they stay small and responsive.


def load_source_slice(site_id: int, start_dt, end_dt) -> pd.DataFrame:
    """Load original selected source-history target rows for one site and date interval."""
    start_ts = pd.Timestamp(start_dt)
    end_ts = pd.Timestamp(end_dt)
    table = source_dataset.to_table(
        columns=["site_id", "datetime", *TARGET_COLUMNS],
        filter=(
            (ds.field("site_id") == int(site_id))
            & (ds.field("datetime") >= pa.scalar(start_ts.to_pydatetime()))
            & (ds.field("datetime") <= pa.scalar(end_ts.to_pydatetime()))
        ),
    )
    source = table.to_pandas().sort_values("datetime").reset_index(drop=True)
    if source.empty:
        return source
    source["datetime"] = pd.to_datetime(source["datetime"])
    source["pv_generation_kW"] = source["underlying_load_kW"] - source["net_load_with_pv_kW"]
    source["battery_storage_kW"] = source["net_load_with_pv_and_battery_kW"] - source["net_load_with_pv_kW"]
    source["battery_net_discharge_kW"] = -source["battery_storage_kW"]
    source["is_negative_underlying"] = source["underlying_load_kW"].lt(NEGATIVE_THRESHOLD_KW)
    return source


def site_timeseries_slice(site_id: int, start_dt, end_dt) -> pd.DataFrame:
    """Return the already loaded post-fill rows for one site and date interval."""
    start_ts = pd.Timestamp(start_dt)
    end_ts = pd.Timestamp(end_dt)
    mask = (
        negative_ts["site_id"].eq(int(site_id))
        & negative_ts["datetime"].between(start_ts, end_ts, inclusive="both")
    )
    return negative_ts.loc[mask].sort_values("datetime").copy()


def worst_window_for_site(site_id: int, hours_each_side: int = 36) -> tuple[pd.Timestamp, pd.Timestamp]:
    """Choose a default drilldown window around the worst negative event for a site."""
    site_rows = negative_ts.loc[negative_ts["site_id"].eq(int(site_id))]
    worst_row = site_rows.loc[site_rows["underlying_load_kW"].idxmin()]
    start_dt = max(site_rows["datetime"].min(), worst_row["datetime"] - pd.Timedelta(hours=hours_each_side))
    end_dt = min(site_rows["datetime"].max(), worst_row["datetime"] + pd.Timedelta(hours=hours_each_side))
    return start_dt, end_dt


def make_site_drilldown(site_id: int, start_dt=None, end_dt=None, include_source_points: bool = True):
    """Build an interactive Plotly time-series diagnostic for one household."""
    if not PLOTLY_AVAILABLE:
        print("Install plotly to render the per-household drilldown.")
        return None

    if start_dt is None or end_dt is None:
        start_dt, end_dt = worst_window_for_site(site_id)
    start_ts = pd.Timestamp(start_dt)
    end_ts = pd.Timestamp(end_dt)

    site_df = site_timeseries_slice(site_id, start_ts, end_ts)
    if site_df.empty:
        print(f"No post-fill rows found for site {site_id} between {start_ts} and {end_ts}.")
        return None

    meta = ranked_negative.loc[ranked_negative["site_id"].eq(int(site_id))].iloc[0]
    fig = go.Figure()
    for column in COMPONENT_COLUMNS:
        fig.add_trace(
            go.Scatter(
                x=site_df["datetime"],
                y=site_df[column],
                mode="lines",
                name=COMPONENT_LABELS[column],
                line={"width": 1.6, "color": COMPONENT_COLORS.get(column)},
            )
        )

    negative_points = site_df.loc[site_df["is_negative_underlying"]]
    fig.add_trace(
        go.Scatter(
            x=negative_points["datetime"],
            y=negative_points["underlying_load_kW"],
            mode="markers",
            name="Negative underlying points",
            marker={"color": "firebrick", "size": 6, "opacity": 0.85},
            hovertemplate="%{x}<br>underlying=%{y:.3f} kW<extra></extra>",
        )
    )

    if include_source_points:
        # Source points expose the original complete observations before the post-fill grid was created.
        source_df = load_source_slice(site_id, start_ts, end_ts)
        if "is_negative_underlying" in source_df:
            source_negative = source_df.loc[source_df["is_negative_underlying"]]
        else:
            source_negative = pd.DataFrame()
        if not source_negative.empty:
            fig.add_trace(
                go.Scatter(
                    x=source_negative["datetime"],
                    y=source_negative["underlying_load_kW"],
                    mode="markers",
                    name="Source negative observations",
                    marker={"symbol": "x", "color": "black", "size": 7},
                    hovertemplate="source %{x}<br>underlying=%{y:.3f} kW<extra></extra>",
                )
            )

    fig.add_hline(y=NEGATIVE_THRESHOLD_KW, line_dash="dash", line_color="firebrick")
    fig.add_hline(y=0, line_width=0.8, line_color="rgba(60,60,60,0.65)")
    fig.update_layout(
        title=(
            f"Site {int(site_id)} drilldown | rank {int(meta['rank_by_negative_count'])} | "
            f"negative {meta['negative_pct']:.2f}% | min {meta['min_underlying_load_kW']:.2f} kW"
        ),
        height=560,
        xaxis_title="Datetime, fixed AEST",
        yaxis_title="Power (kW)",
        legend_title_text="Series",
    )
    fig.show()

    worst_events = (
        site_df.nsmallest(20, "underlying_load_kW")[[
            "datetime",
            "underlying_load_kW",
            "net_load_with_pv_and_battery_kW",
            "pv_generation_kW",
            "battery_storage_kW",
            "net_load_with_pv_kW",
        ]]
        .round(3)
        .reset_index(drop=True)
    )
    display(worst_events)
    return fig


def make_site_weekly_profile(site_id: int):
    """Build a typical weekly profile for one household."""
    if not PLOTLY_AVAILABLE:
        print("Install plotly to render the site weekly profile.")
        return None
    site_df = negative_ts.loc[negative_ts["site_id"].eq(int(site_id))]
    site_week = (
        site_df.groupby("week_slot_5min", observed=True)[COMPONENT_COLUMNS]
        .mean()
        .reindex(range(7 * 288))
        .reset_index()
    )
    fig = go.Figure()
    for column in COMPONENT_COLUMNS:
        fig.add_trace(
            go.Scatter(
                x=site_week["week_slot_5min"],
                y=site_week[column],
                mode="lines",
                name=COMPONENT_LABELS[column],
                line={"width": 2, "color": COMPONENT_COLORS.get(column)},
            )
        )
    for day in range(8):
        fig.add_vline(x=day * 288, line_width=0.7, line_color="rgba(120,120,120,0.35)")
    fig.add_hline(y=NEGATIVE_THRESHOLD_KW, line_dash="dash", line_color="firebrick")
    fig.update_layout(
        title=f"Site {int(site_id)} typical weekly component profile",
        height=500,
        xaxis_title="Typical week",
        yaxis_title="Power (kW)",
    )
    fig.update_xaxes(tickmode="array", tickvals=WEEK_SLOT_TICKS, ticktext=WEEK_SLOT_LABELS)
    fig.show()
    return fig

In [ ]:
# Purpose: expose a widget-driven drilldown when ipywidgets is available.
# If widgets are unavailable, show a default worst-site example so the cell still produces useful output.

site_options = [
    (
        f"rank {int(row.rank_by_negative_count):02d} | site {int(row.site_id)} | neg {row.negative_pct:.1f}% | min {row.min_underlying_load_kW:.1f} kW",
        int(row.site_id),
    )
    for row in ranked_negative.itertuples(index=False)
]

default_site_id = site_options[0][1]
default_start, default_end = worst_window_for_site(default_site_id)

if PLOTLY_AVAILABLE and WIDGETS_AVAILABLE:
    available_dates = pd.date_range(
        negative_ts["datetime"].min().normalize(),
        negative_ts["datetime"].max().normalize(),
        freq="D",
    )
    date_options = [(date.strftime("%Y-%m-%d"), date.date()) for date in available_dates]
    default_start_date = default_start.normalize().date()
    default_end_date = default_end.normalize().date()
    start_index = max(0, available_dates.get_indexer([pd.Timestamp(default_start_date)], method="nearest")[0])
    end_index = max(start_index, available_dates.get_indexer([pd.Timestamp(default_end_date)], method="nearest")[0])

    site_dropdown = widgets.Dropdown(options=site_options, value=default_site_id, description="Site", layout=widgets.Layout(width="650px"))
    date_range = widgets.SelectionRangeSlider(
        options=date_options,
        index=(int(start_index), int(end_index)),
        description="Dates",
        continuous_update=False,
        layout=widgets.Layout(width="900px"),
    )
    source_toggle = widgets.Checkbox(value=True, description="Overlay source negative observations")

    def render_widget_drilldown(site_id, date_range, include_source_points):
        start_date, end_date = date_range
        start_ts = pd.Timestamp(start_date)
        end_ts = pd.Timestamp(end_date) + pd.Timedelta(days=1) - pd.Timedelta(minutes=5)
        make_site_drilldown(site_id, start_ts, end_ts, include_source_points)
        make_site_weekly_profile(site_id)

    controls = widgets.VBox([site_dropdown, date_range, source_toggle])
    output = widgets.interactive_output(
        render_widget_drilldown,
        {"site_id": site_dropdown, "date_range": date_range, "include_source_points": source_toggle},
    )
    display(controls, output)
elif PLOTLY_AVAILABLE:
    print("ipywidgets is unavailable, so this cell renders a default worst-site drilldown instead of controls.")
    make_site_drilldown(default_site_id, default_start, default_end, include_source_points=True)
    make_site_weekly_profile(default_site_id)
else:
    print("Install plotly and ipywidgets to enable the per-household interactive drilldown controls.")

## 9. Optional Raw Polarity Drilldown

The processed data already uses the corrected polarity convention, but this optional section can inspect the original raw daily parquet rows for one site and a small date range. Keep the window small: it reads raw parquet files for the selected date span and then displays raw power, circuit polarity, adjusted power, and the reconstructed components.

In [ ]:
# Purpose: define raw-data helpers for a targeted polarity check.
# These functions read only the raw daily parquet files needed for one site and one small time window.

REQUIRED_RAW_CIRCUIT_TYPES = ["ac_load_net", "pv_site_net", "battery_storage"]


def raw_parquet_paths_for_aest_window(start_dt, end_dt) -> list[Path]:
    """Return raw daily parquet files that can overlap a fixed-AEST date window."""
    if not RAW_BESS_DIR.exists():
        return []
    raw_start = pd.Timestamp(start_dt) - pd.Timedelta(hours=FIXED_AEST_OFFSET_HOURS)
    raw_end = pd.Timestamp(end_dt) - pd.Timedelta(hours=FIXED_AEST_OFFSET_HOURS)
    needed_stems = {
        date.strftime("%Y%m%d")
        for date in pd.date_range(raw_start.normalize(), raw_end.normalize(), freq="D")
    }
    return [path for path in sorted(RAW_BESS_DIR.rglob("*.parquet")) if path.stem in needed_stems]


def load_raw_polarity_slice(site_id: int, start_dt, end_dt) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load raw circuit rows and a wide adjusted-power panel for one site/window."""
    if not CIRCUIT_META_PATH.exists():
        raise FileNotFoundError(f"Circuit metadata not found: {CIRCUIT_META_PATH}")

    circuit_meta = pd.read_csv(CIRCUIT_META_PATH)
    site_circuits = circuit_meta.loc[
        circuit_meta["site_id"].astype("int64").eq(int(site_id))
        & circuit_meta["circuit_type"].isin(REQUIRED_RAW_CIRCUIT_TYPES)
    ].copy()
    if site_circuits.empty:
        raise ValueError(f"No required raw circuit metadata found for site {site_id}.")

    site_circuits["circuit_id"] = site_circuits["circuit_id"].astype("int64")
    site_circuits["circuit_polarity"] = pd.to_numeric(site_circuits["circuit_polarity"], errors="coerce").fillna(1.0)
    circuit_ids = site_circuits["circuit_id"].tolist()

    start_aest = pd.Timestamp(start_dt)
    end_aest = pd.Timestamp(end_dt)
    raw_start = start_aest - pd.Timedelta(hours=FIXED_AEST_OFFSET_HOURS)
    raw_end = end_aest - pd.Timedelta(hours=FIXED_AEST_OFFSET_HOURS)

    parts = []
    for parquet_path in raw_parquet_paths_for_aest_window(start_aest, end_aest):
        daily = pd.read_parquet(parquet_path, columns=["circuit_id", "t_stamp", "power"])
        if daily.empty:
            continue
        daily["circuit_id"] = daily["circuit_id"].astype("int64")
        daily = daily.loc[daily["circuit_id"].isin(circuit_ids)].copy()
        if daily.empty:
            continue
        daily["t_stamp"] = pd.to_datetime(daily["t_stamp"], errors="coerce")
        daily = daily.loc[daily["t_stamp"].between(raw_start, raw_end, inclusive="both")].copy()
        if not daily.empty:
            parts.append(daily)

    if not parts:
        return pd.DataFrame(), pd.DataFrame()

    raw_rows = pd.concat(parts, ignore_index=True)
    raw_rows = raw_rows.merge(
        site_circuits[["circuit_id", "circuit_type", "circuit_polarity"]],
        on="circuit_id",
        how="left",
    )
    raw_rows["raw_power_kW"] = pd.to_numeric(raw_rows["power"], errors="coerce") / 1000.0
    raw_rows["adjusted_power_kW"] = raw_rows["raw_power_kW"] * raw_rows["circuit_polarity"]
    raw_rows["datetime"] = raw_rows["t_stamp"] + pd.Timedelta(hours=FIXED_AEST_OFFSET_HOURS)
    raw_rows = raw_rows.loc[raw_rows["datetime"].between(start_aest, end_aest, inclusive="both")].copy()

    adjusted_panel = (
        raw_rows.groupby(["datetime", "circuit_type"], observed=True)["adjusted_power_kW"]
        .sum()
        .unstack("circuit_type")
        .reset_index()
    )
    for column in REQUIRED_RAW_CIRCUIT_TYPES:
        if column not in adjusted_panel.columns:
            adjusted_panel[column] = np.nan
    adjusted_panel["underlying_from_raw_kW"] = (
        adjusted_panel["ac_load_net"] + adjusted_panel["pv_site_net"] - adjusted_panel["battery_storage"]
    )
    adjusted_panel["pv_generation_from_raw_kW"] = adjusted_panel["pv_site_net"]
    adjusted_panel["battery_storage_from_raw_kW"] = adjusted_panel["battery_storage"]
    adjusted_panel["is_negative_underlying"] = adjusted_panel["underlying_from_raw_kW"].lt(NEGATIVE_THRESHOLD_KW)

    return raw_rows.sort_values(["datetime", "circuit_type"]), adjusted_panel.sort_values("datetime")


def plot_raw_polarity_slice(site_id: int, start_dt, end_dt):
    """Render raw and adjusted circuit signals for one site/date range."""
    raw_rows, adjusted_panel = load_raw_polarity_slice(site_id, start_dt, end_dt)
    if raw_rows.empty:
        print(f"No raw rows found for site {site_id} between {start_dt} and {end_dt}.")
        return None

    display(raw_rows[[
        "datetime",
        "circuit_type",
        "circuit_id",
        "circuit_polarity",
        "raw_power_kW",
        "adjusted_power_kW",
    ]].head(30).round(4))

    if not PLOTLY_AVAILABLE:
        display(adjusted_panel.head(30).round(4))
        print("Install plotly to render the raw polarity figure.")
        return adjusted_panel

    long_adjusted = adjusted_panel.melt(
        id_vars=["datetime", "is_negative_underlying"],
        value_vars=[
            "ac_load_net",
            "pv_site_net",
            "battery_storage",
            "underlying_from_raw_kW",
        ],
        var_name="series",
        value_name="power_kW",
    )
    fig = px.line(
        long_adjusted,
        x="datetime",
        y="power_kW",
        color="series",
        title=f"Raw polarity-adjusted circuit slice for site {site_id}",
        labels={"datetime": "Datetime, fixed AEST", "power_kW": "Power (kW)", "series": "Series"},
        height=560,
    )
    negative_points = adjusted_panel.loc[adjusted_panel["is_negative_underlying"]]
    fig.add_trace(
        go.Scatter(
            x=negative_points["datetime"],
            y=negative_points["underlying_from_raw_kW"],
            mode="markers",
            name="Negative underlying from raw slice",
            marker={"color": "firebrick", "size": 7},
        )
    )
    fig.add_hline(y=NEGATIVE_THRESHOLD_KW, line_dash="dash", line_color="firebrick")
    fig.add_hline(y=0, line_width=0.8, line_color="rgba(60,60,60,0.65)")
    fig.show()
    return adjusted_panel

In [ ]:
# Purpose: provide a safe example for raw polarity inspection without reading raw data automatically.
# Set RUN_RAW_POLARITY_EXAMPLE to True after choosing a small site/date window.

RUN_RAW_POLARITY_EXAMPLE = False

raw_example_site = int(ranked_negative.iloc[0]["site_id"])
raw_example_start, raw_example_end = worst_window_for_site(raw_example_site, hours_each_side=6)

print("Raw polarity example is prepared but not executed by default.")
print(f"Example site: {raw_example_site}")
print(f"Example window: {raw_example_start} to {raw_example_end}")
print("To run it, set RUN_RAW_POLARITY_EXAMPLE = True and re-run this cell.")

if RUN_RAW_POLARITY_EXAMPLE:
    plot_raw_polarity_slice(raw_example_site, raw_example_start, raw_example_end)

## 10. Interpretation Notes

This final section turns the diagnostics into a compact checklist for interpreting causes. It does not change the processing formula; it simply highlights which signal should be inspected first for each household.

In [ ]:
# Purpose: summarize the diagnostic hints and expose the most severe examples for manual review.
# Use this as a triage list before changing any polarity or formula assumptions.

hint_counts = (
    ranked_negative.groupby("screening_hint", observed=True)
    .agg(
        households=("site_id", "size"),
        total_negative_rows=("negative_count", "sum"),
        median_negative_pct=("negative_pct", "median"),
        worst_min_underlying_kW=("min_underlying_load_kW", "min"),
    )
    .sort_values("total_negative_rows", ascending=False)
    .reset_index()
)

severe_examples = ranked_negative.nsmallest(15, "min_underlying_load_kW")[[
    "rank_by_negative_count",
    "selection_rank",
    "site_id",
    "negative_count",
    "negative_pct",
    "min_underlying_load_kW",
    "median_corrected_net_kW",
    "median_pv_generation_kW",
    "median_battery_storage_kW",
    "screening_hint",
]].round(3)

print("Screening hint summary:")
display(hint_counts.round(3))

print("Most severe minimum-underlying examples:")
display(severe_examples)

print("Interpretation checklist:")
print("1. If corrected net load is strongly negative during negative underlying intervals, inspect ac_load_net/export behavior first.")
print("2. If PV generation is often negative or large at implausible times, inspect pv_site_net polarity and night offsets.")
print("3. If positive battery_storage dominates, inspect whether charging should be subtracted in the selected formula.")
print("4. Use the raw polarity drilldown on small windows before changing global signal assumptions.")